# Sensoren, Impacts und Virtual Points setzen

Zwei Buchsen verbinden Subframe und ControlArm. Jede wird zu einem Virtual Point.

| Grouping | |
|---|---|
| `1` | Buchse 1 — Interface, beide Bauteile |
| `2` | Buchse 2 — Interface, beide Bauteile |
| `10` | Antwort-Sensor, Subframe |
| `20` | Anregungs-Impact, ControlArm |

`grouping=` gilt fuer alle Punkte, die *nach* dem Zellenaufruf geklickt werden.

In [1]:
import pyfbs
import pandas as pd
pos_xlsx = r"./Subframe_Control_Arm/Measurements/measurements.xlsx"
MK_ControlArm = pyfbs.mck.Model.from_ansys(r"./Subframe_Control_Arm/FEM/ControlArm.rst", r"./Subframe_Control_Arm/FEM/ControlArm.full", no_modes=100, mesh_scale=1)
MK_Subframe = pyfbs.mck.Model.from_ansys(r"./Subframe_Control_Arm/FEM/Subframe.rst", r"./Subframe_Control_Arm/FEM/Subframe.full", no_modes=100, mesh_scale=1)

In [2]:
from pyfbs.display import DynamicPosition
from pyfbs.utility import rotation_matrix_from_vectors


def place_impacts(view, df):
    """Gespeicherte Impacts exakt an Position und Richtung aus df in den View setzen.

    Entspricht View3D.imp_callback, aber ohne Snapping. imp_callback schiesst einen
    Strahl von der aktuellen Kamera durch den Punkt, setzt den Impact auf die erste
    getroffene Flaeche und ersetzt die Richtung durch deren Normale -- beim Neuladen
    wandern Impacts auf verdeckten Flaechen dadurch weg. Vorher
    view.add_imp_dynamic(mesh, grouping=g) aufrufen, das setzt mesh, size und grouping.
    """
    for _, row in df.iterrows():
        point = row[["Position_1", "Position_2", "Position_3"]].to_numpy(float)
        direction = row[["Direction_1", "Direction_2", "Direction_3"]].to_numpy(float)
        # Jeder dynamische Punkt belegt 4 Sphere-Widgets; i ist sein laufender Index
        i = len(view.all_accs_dynamic) + len(view.all_imps_dynamic) + len(view.all_vps_dynamic)
        imp, _ = view.add_impact([view.size / 2] * 3, direction, size=view.size)
        rot = rotation_matrix_from_vectors(direction, [0, 0, 1]).T
        gg = DynamicPosition([imp], view.plot, i, mesh=view.mesh, size=view.size, rot=rot,
                             snap_outward=False, fixed_rotation=view.fixed_rotation,
                             toggle="impact", grouping=view.grouping)
        view.plot.add_sphere_widget(gg.callback, center=gg.points,
                                    color=["k", "r", "g", "b"], radius=view.size / 15)
        gg.translate(point)          # ohne snap: reine Verschiebung, Richtung bleibt
        gg.turn_on = True
        view.all_imps_dynamic.append(gg)

## ControlArm

In [7]:
view_CA = pyfbs.display.View3D(show_origin=True)
surf_CA = view_CA.add_mesh(MK_ControlArm.mesh, name="mesh", cmap=None, color="#D3E0EC", show_edges=True, edge_color="#9DB2C4")

C:\Users\yanni\Desktop\TUM\Bachelorarbeit\Fast_numerical_solution_for_nonlinear_FE_dynamics\pyFBS_clone\pyFBS\pyfbs\display\display.py:2136: PyVistaDeprecationWarning: ``Plotter.sphere_widgets`` is deprecated; use ``Plotter.widgets.sphere_widgets`` instead.
  self.p.sphere_widgets[self.N + 0].SetCenter(point)
C:\Users\yanni\Desktop\TUM\Bachelorarbeit\Fast_numerical_solution_for_nonlinear_FE_dynamics\pyFBS_clone\pyFBS\pyfbs\display\display.py:2141: PyVistaDeprecationWarning: ``Plotter.sphere_widgets`` is deprecated; use ``Plotter.widgets.sphere_widgets`` instead.
  self.p.sphere_widgets[self.N + k + 1].SetCenter(
C:\Users\yanni\Desktop\TUM\Bachelorarbeit\Fast_numerical_solution_for_nonlinear_FE_dynamics\pyFBS_clone\pyFBS\pyfbs\display\display.py:2177: PyVistaDeprecationWarning: ``Plotter.sphere_widgets`` is deprecated; use ``Plotter.widgets.sphere_widgets`` instead.
  self.p.sphere_widgets[self.N + i].SetCenter(
C:\Users\yanni\Desktop\TUM\Bachelorarbeit\Fast_numerical_solution_for_nonli

Optional: gespeicherte Punkte weiterbearbeiten.

In [4]:
# add_*_dynamic ignoriert die Grouping-Spalte von predefined: jeder geladene Punkt
# bekommt das grouping-Argument des Aufrufs. Daher jede Gruppe einzeln laden.
# Impacts ueber place_impacts, damit sie beim Laden nicht neu gesnappt werden.
with pd.ExcelFile(pos_xlsx) as xl:
    if "Sensors_ControlArm" in xl.sheet_names:
        sens, imps = xl.parse("Sensors_ControlArm"), xl.parse("Impacts_ControlArm")
        for g, grp in sens.groupby("Grouping", sort=False):
            view_CA.add_acc_dynamic(surf_CA, predefined=grp, grouping=int(g))
        for g, grp in imps.groupby("Grouping", sort=False):
            view_CA.add_imp_dynamic(surf_CA, grouping=int(g))
            place_impacts(view_CA, grp)

NameError: name 'view_CA' is not defined

#### Sensoren

In [8]:
view_CA.add_acc_dynamic(surf_CA, grouping=1)

In [9]:
view_CA.add_acc_dynamic(surf_CA, grouping=2)

#### Impacts

In [10]:
view_CA.add_imp_dynamic(surf_CA, grouping=1)

In [11]:
view_CA.add_imp_dynamic(surf_CA, grouping=2)

#### Anregungspunkt

In [12]:
view_CA.add_imp_dynamic(surf_CA, grouping=20)

## Subframe

In [13]:
view_SF = pyfbs.display.View3D(show_origin=True)
surf_SF = view_SF.add_mesh(MK_Subframe.mesh, name="mesh", cmap=None, color="#D3E0EC", show_edges=True, edge_color="#9DB2C4")

Optional: gespeicherte Punkte weiterbearbeiten.

In [5]:
# add_*_dynamic ignoriert die Grouping-Spalte von predefined: jeder geladene Punkt
# bekommt das grouping-Argument des Aufrufs. Daher jede Gruppe einzeln laden.
# Impacts ueber place_impacts, damit sie beim Laden nicht neu gesnappt werden.
with pd.ExcelFile(pos_xlsx) as xl:
    if "Sensors_Subframe" in xl.sheet_names:
        sens, imps = xl.parse("Sensors_Subframe"), xl.parse("Impacts_Subframe")
        for g, grp in sens.groupby("Grouping", sort=False):
            view_SF.add_acc_dynamic(surf_SF, predefined=grp, grouping=int(g))
        for g, grp in imps.groupby("Grouping", sort=False):
            view_SF.add_imp_dynamic(surf_SF, grouping=int(g))
            place_impacts(view_SF, grp)

NameError: name 'view_SF' is not defined

#### Sensoren

In [14]:
view_SF.add_acc_dynamic(surf_SF, grouping=1)

In [15]:
view_SF.add_acc_dynamic(surf_SF, grouping=2)

#### Impacts

In [16]:
view_SF.add_imp_dynamic(surf_SF, grouping=1)

In [17]:
view_SF.add_imp_dynamic(surf_SF, grouping=2)

#### Antwortpunkt

In [18]:
view_SF.add_acc_dynamic(surf_SF, grouping=10)

## Auslesen

In [19]:
df_acc_CA = view_CA.get_acc_data()
df_chn_CA = pyfbs.utility.generate_channels_from_sensors(df_acc_CA)
df_imp_CA = view_CA.get_imp_data()
df_chn_CA

,Name,Description,Quantity,Grouping,Position_1,Position_2,Position_3,Direction_1,Direction_2,Direction_3
0,Sensor 1x,None,None,1,945.261047,-248.958298,324.053925,0.998467,0.0357,0.042301
1,Sensor 1y,None,None,1,945.261047,-248.958298,324.053925,-0.02923,0.989035,-0.144762
2,Sensor 1z,None,None,1,945.261047,-248.958298,324.053925,-0.047005,0.143304,0.988562
3,Sensor 2x,None,None,1,985.914917,-260.348938,310.899597,0.992633,-0.027538,0.11799
4,Sensor 2y,None,None,1,985.914917,-260.348938,310.899597,-0.026698,0.900179,0.4347
5,Sensor 2z,None,None,1,985.914917,-260.348938,310.899597,-0.118182,-0.434648,0.892813
6,Sensor 3x,None,None,1,975.830322,-262.946594,345.01474,0.99801,-0.052121,-0.035488
7,Sensor 3y,None,None,1,975.830322,-262.946594,345.01474,-0.037602,-0.943729,0.328575
8,Sensor 3z,None,None,1,975.830322,-262.946594,345.01474,-0.050617,-0.326586,-0.943811
9,Sensor 4x,None,None,2,690.088013,-247.527237,316.384125,0.999756,-0.02196,-0.002285


In [20]:
df_acc_SF = view_SF.get_acc_data()
df_chn_SF = pyfbs.utility.generate_channels_from_sensors(df_acc_SF)
df_imp_SF = view_SF.get_imp_data()
df_chn_SF

C:\Users\yanni\Desktop\TUM\Bachelorarbeit\Fast_numerical_solution_for_nonlinear_FE_dynamics\pyFBS_clone\pyFBS\pyfbs\display\display.py:2037: UserWarning: Gimbal lock detected. Setting third angle to zero since it is not possible to uniquely determine all angles.
  orientation = r.as_euler('xyz', degrees=True)


,Name,Description,Quantity,Grouping,Position_1,Position_2,Position_3,Direction_1,Direction_2,Direction_3
0,Sensor 1x,None,None,1,985.0,-288.054962,303.43512,0.0,0.0,-1.0
1,Sensor 1y,None,None,1,985.0,-288.054962,303.43512,0.0,1.0,0.0
2,Sensor 1z,None,None,1,985.0,-288.054962,303.43512,1.0,0.0,0.0
3,Sensor 2x,None,None,1,945.0,-296.975891,337.793243,0.0,0.0,1.0
4,Sensor 2y,None,None,1,945.0,-296.975891,337.793243,-0.0,1.0,0.0
5,Sensor 2z,None,None,1,945.0,-296.975891,337.793243,-1.0,-0.0,0.0
6,Sensor 3x,None,None,1,962.167297,-252.295792,302.856354,0.997992,-0.06294,-0.007117
7,Sensor 3y,None,None,1,962.167297,-252.295792,302.856354,-0.023297,-0.469227,0.88277
8,Sensor 3z,None,None,1,962.167297,-252.295792,302.856354,-0.058901,-0.880832,-0.469751
9,Sensor 4x,None,None,2,718.938782,-282.461853,355.985443,0.0,0.0,-1.0


## Virtual Points

Buchsenmitten aus den Koordinatensystemen der SLDPRT.

In [21]:
# pyfbs.utility.generate_vp_from_position ist mit pandas 3 defekt (errors='ignore'),
# daher hier direkt gebaut. Achsen global, also keine Drehung.
vp_pos = {1: [965.00, -266.5, 327.5],     # KoordinatensystemA1
          2: [677.94, -271.0, 335.0]}     # KoordinatensystemB1

def vp_frame(desc, quantity):
    rows = []
    for g, p in vp_pos.items():
        for k, d in enumerate(desc):
            rows.append({"Name": f"VP{g}_{d}", "Description": d, "Quantity": quantity[k],
                         "Grouping": g,
                         "Position_1": p[0], "Position_2": p[1], "Position_3": p[2],
                         "Direction_1": float(k % 3 == 0),
                         "Direction_2": float(k % 3 == 1),
                         "Direction_3": float(k % 3 == 2)})
    return pd.DataFrame(rows)

df_vp    = vp_frame(["ux", "uy", "uz", "rx", "ry", "rz"],
                    ["Acceleration"] * 3 + ["Rotational acceleration"] * 3)
df_vpref = vp_frame(["fx", "fy", "fz", "mx", "my", "mz"],
                    ["Force"] * 3 + ["Moment"] * 3)
df_vp

,Name,Description,Quantity,Grouping,Position_1,Position_2,Position_3,Direction_1,Direction_2,Direction_3
0,VP1_ux,ux,Acceleration,1,965.00,-266.5,327.5,1.0,0.0,0.0
1,VP1_uy,uy,Acceleration,1,965.00,-266.5,327.5,0.0,1.0,0.0
2,VP1_uz,uz,Acceleration,1,965.00,-266.5,327.5,0.0,0.0,1.0
3,VP1_rx,rx,Rotational acceleration,1,965.00,-266.5,327.5,1.0,0.0,0.0
4,VP1_ry,ry,Rotational acceleration,1,965.00,-266.5,327.5,0.0,1.0,0.0
5,VP1_rz,rz,Rotational acceleration,1,965.00,-266.5,327.5,0.0,0.0,1.0
6,VP2_ux,ux,Acceleration,2,677.94,-271.0,335.0,1.0,0.0,0.0
7,VP2_uy,uy,Acceleration,2,677.94,-271.0,335.0,0.0,1.0,0.0
8,VP2_uz,uz,Acceleration,2,677.94,-271.0,335.0,0.0,0.0,1.0
9,VP2_rx,rx,Rotational acceleration,2,677.94,-271.0,335.0,1.0,0.0,0.0


## Speichern

In [22]:
with pd.ExcelWriter(pos_xlsx) as writer:
    df_acc_SF.to_excel(writer, sheet_name="Sensors_Subframe", index=False)
    df_chn_SF.to_excel(writer, sheet_name="Channels_Subframe", index=False)
    df_imp_SF.to_excel(writer, sheet_name="Impacts_Subframe", index=False)
    df_acc_CA.to_excel(writer, sheet_name="Sensors_ControlArm", index=False)
    df_chn_CA.to_excel(writer, sheet_name="Channels_ControlArm", index=False)
    df_imp_CA.to_excel(writer, sheet_name="Impacts_ControlArm", index=False)
    df_vp.to_excel(writer, sheet_name="VP_Channels", index=False)
    df_vpref.to_excel(writer, sheet_name="VP_RefChannels", index=False)

## Gespeicherte Tabelle laden

Nur zum Pruefen: liest die Tabelle direkt, ohne 3D-Views. Danach VPT und Ansehen.

In [2]:
with pd.ExcelFile(pos_xlsx) as xl:
    df_chn_SF, df_imp_SF = xl.parse("Channels_Subframe"), xl.parse("Impacts_Subframe")
    df_chn_CA, df_imp_CA = xl.parse("Channels_ControlArm"), xl.parse("Impacts_ControlArm")
    df_vp, df_vpref = xl.parse("VP_Channels"), xl.parse("VP_RefChannels")

## VPT

Konsistenz nahe 1 = die Punkte sitzen auf einem Bereich, der sich starr mitbewegt.

In [ ]:
df_chn_SF = MK_Subframe.update_locations_df(df_chn_SF)
df_imp_SF = MK_Subframe.update_locations_df(df_imp_SF)
MK_Subframe.frf_synth(df_chn_SF, df_imp_SF, f_start=1, f_end=1200, f_resolution=2, modal_damping=0.01)
vpt_SF = pyfbs.interface.VPT(df_chn_SF, df_imp_SF, df_vp, df_vpref)
vpt_SF.apply_vpt(MK_Subframe.freq, MK_Subframe.frf)

In [ ]:
df_chn_CA = MK_ControlArm.update_locations_df(df_chn_CA)
df_imp_CA = MK_ControlArm.update_locations_df(df_imp_CA)
MK_ControlArm.frf_synth(df_chn_CA, df_imp_CA, f_start=1, f_end=1200, f_resolution=2, modal_damping=0.01)
vpt_CA = pyfbs.interface.VPT(df_chn_CA, df_imp_CA, df_vp, df_vpref)
vpt_CA.apply_vpt(MK_ControlArm.freq, MK_ControlArm.frf)

In [ ]:
for g in [1, 2]:
    for name, vpt in [("Subframe", vpt_SF), ("ControlArm", vpt_CA)]:
        vpt.consistency([g], [g])
        print(f"Buchse {g}  {name:<11} Sensor {vpt.overall_sensor.mean():.3f}   "f"Impact {vpt.overall_impact.mean():.3f}")

## Ansehen

In [3]:
vis_SF = pyfbs.display.View3D(show_origin=False)
vis_SF.add_mesh(MK_Subframe.mesh, name="SF", cmap=None, color="#D3E0EC", opacity=0.4)
vis_SF.show_chn(df_chn_SF, size=8)
vis_SF.show_imp(df_imp_SF, size=12)
vis_SF.show_vp(df_vp, size=6)

C:\Users\yanni\Desktop\TUM\Bachelorarbeit\Fast_numerical_solution_for_nonlinear_FE_dynamics\pyFBS_clone\pyFBS\pyfbs\display\display.py:559: PyVistaFutureWarning: The default value of `algorithm` for the filter
`UnstructuredGrid.extract_surface` will change in the future. It currently defaults to
`'dataset_surface'`, but will change to `None`. Explicitly set the `algorithm` keyword to
silence this warning.
  _mesh = mesh.extract_surface()


In [4]:
vis_CA = pyfbs.display.View3D(show_origin=False)
vis_CA.add_mesh(MK_ControlArm.mesh, name="CA", cmap=None, color="#E8D5C4", opacity=0.4)
vis_CA.show_chn(df_chn_CA, size=8)
vis_CA.show_imp(df_imp_CA, size=12)
vis_CA.show_vp(df_vp, size=6)